In [1]:
include(dirname(dirname(pwd()))*"\\src\\TuLiPa.jl");
using .TuLiPa
using Dates, DataFrames, CSV, JSON, Plots, JuMP, HiGHS
include(dirname(dirname(pwd())) * raw"\demos\other\flowbased_utils.jl")
include(dirname(dirname(pwd())) * raw"\demos\other\flowbased_utils2.jl")

make_connection (generic function with 2 methods)

In [2]:
emps_csv_path = "./assets/emps_production_demand_nordics2.csv"
df_emps = CSV.read(emps_csv_path, DataFrame)
emps_dict = Dict{String, Dict{String, AbstractFloat}}()  # Initialize empty dataframe

# Loop through the dataframe to create a dict
for row in eachrow(df_emps)
    emps_dict[row.emps] = Dict("Power" => row.power_cost, "Power_cap" => row.power_cap, 
    "Demand" => row.demand_vol)
end

In [3]:
fix_names = Dict([
    "SE1" => "SVER-SE1", 
    "SE2" => "SVER-SE2",
    "SE3" => "SVER-SE3",
    "SE4" => "SVER-SE4",
    "DK2" => "DANM-DK2",
    "FI" => "FINLAND",
]
)

function rename_emps!(df::DataFrame, columns::Vector{Symbol}, fix_names::Dict{String, String})
    # Oppdater verdier i spesifiserte kolonner
    for col in columns
        df[!, col] .= map(x -> get(fix_names, x, x), df[!, col])
    end
    # Oppdater kolonnenavn
    rename!(df, Symbol.(map(x -> get(fix_names, String(x), String(x)), names(df))))
end

ptdf_csv_path = "./assets/ptdfs_new2.csv"
df_flowbased_grid = CSV.read(ptdf_csv_path, DataFrame)

# Bruk funksjonen på DataFrame
rename_emps!(df_flowbased_grid, [:emps_area0, :emps_area1], fix_names);



df_flowbased = process_ptdf_matrix(df_grid, false)

In [4]:
elements = create_elements(emps_dict, df_flowbased_grid);
transm = [e for e in elements if is_transmission(e)];

Dict{String, Dict{String, AbstractFloat}}("TELEMARK" => Dict("Power" => 50.0, "Power_cap" => 1000.0, "Demand" => 50.0), "SORLAND" => Dict("Power" => 60.0, "Power_cap" => 1000.0, "Demand" => 80.0), "SVARTISEN" => Dict("Power" => 30.0, "Power_cap" => 1000.0, "Demand" => 10.0), "SVER-SE2" => Dict("Power" => 10.0, "Power_cap" => 1000.0, "Demand" => 40.0), "FINLAND" => Dict("Power" => 100.0, "Power_cap" => 1000.0, "Demand" => 100.0), "MOERE" => Dict("Power" => 40.0, "Power_cap" => 1000.0, "Demand" => 40.0), "OSTLAND" => Dict("Power" => 90.0, "Power_cap" => 1000.0, "Demand" => 120.0), "SVER-SE1" => Dict("Power" => 10.0, "Power_cap" => 1000.0, "Demand" => 40.0), "VESTMIDT" => Dict("Power" => 10.0, "Power_cap" => 1000.0, "Demand" => 50.0), "HELGELAND" => Dict("Power" => 5.0, "Power_cap" => 1000.0, "Demand" => 30.0), "HALLINGDAL" => Dict("Power" => 50.0, "Power_cap" => 1000.0, "Demand" => 30.0), "SVER-SE3" => Dict("Power" => 80.0, "Power_cap" => 1000.0, "Demand" => 50.0), "DANM-DK2" => Dict("Po

In [5]:
df_ptdf = process_ptdf_matrix(df_flowbased_grid, false);
#modelobjects = getmodelobjects(elements);

In [6]:
T = DataElement
flow_based = make_flowbased(df_ptdf, transm)
elements = vcat(elements, flow_based);

In [7]:
#modelobjects = TuLiPa.getmodelobjects(elements)

In [8]:
elements = Vector{T}(elements);


In [9]:
modelobjects = TuLiPa.getmodelobjects(elements);

In [10]:
mymodel = JuMP.Model(HiGHS.Optimizer)
prob = TuLiPa.JuMP_Prob(modelobjects, mymodel)
datatime = getisoyearstart(2023)
scenariotime = getisoyearstart(1981)
prob_time = TuLiPa.TwoTime(datatime, scenariotime)  

update!(prob, prob_time)
solve!(prob)

In [13]:
steps = 1
problem = prob
resultobjects = collect(values(modelobjects))
numperiods_powerhorizon = 1
numperiods_hydrohorizon = 1
periodduration_power = Day(1)
t = 1
includeexogenprice=true

prices, rhstermvalues, production, consumption, hydrolevels, batterylevels, powerbalances, rhsterms, rhstermbalances, plants, plantbalances, plantarrows, demands, demandbalances, demandarrows, hydrostorages, batterystorages  = init_results(steps, problem, modelobjects, resultobjects, numperiods_powerhorizon, numperiods_hydrohorizon, periodduration_power, t, includeexogenprice);

In [14]:
println(prices)
println(production)
println(consumption)
println(hydrolevels)

[150.0 100.0 50.0 10.0 90.0 90.0 5.0 20.0 -6.507936506772239 30.0 20.0 40.0 50.0 29.351851851335596 100.0 5.0 80.0 5.0 10.0 60.0 50.0 50.0 31.34920634911532]
[0.02797864814745314 0.10018518518466392 0.0014026537040310405 0.0 0.0 0.12018518518398937 0.08135643517990553 0.08 0.03277777778278427 0.01555555555795194 0.0 0.002932388890588764 0.08814814813848713 0.0293271296280249 0.0 0.049999999999999996 0.0 0.0 0.0 0.04999999999988076 0.0 0.1338888888878505 0.0 0.0 0.005725055555748586 0.08430371295882248 0.0 0.03574439814817738 0.0 0.06981481481601064 0.04808811851683296 0.09333333333153816 0.0 0.0 0.032388762961289556 0.14648148147549414 0.0 0.006731074073815908 0.07981481481588003 0.0 0.10096938703261304 0.0 0.020925925930203426 0.053377787032571615 0.0 0.0 0.012592592594215116 0.0 0.0 0.09000000000000001 0.034743253699298864 0.013948974070026097 0.014255601851822617 0.04296296296730614 0.02314814814812063 0.03722222222316837 0.025910240739868556 0.024770361105915514 0.0 0.0 0.006731074